# Veritas PoC on Google Colab

Runs the full fast-vs-slow causal-scoring pipeline on a Colab GPU.

**Before running:** `Runtime > Change runtime type > T4 GPU`

**HF access:** accept the license at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct, then paste your token in cell 4.

**Important:** Do **not** press Stop / Ctrl+C while a cell is running. Each phase reloads the model once (`Loading weights` is normal). Phase 1 takes ~5–15 min on a T4; wait until you see `Saved 10 trajectories` before running the next cells.

In [ ]:
# 1. Confirm GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Clone repo and cd into it
import os

if os.path.basename(os.getcwd()) != 'algoverse-PoC':
    if not os.path.isdir('algoverse-PoC'):
        !git clone https://github.com/abdelmagid07/algoverse-PoC.git
    %cd algoverse-PoC
!git pull --ff-only

In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 4. Hugging Face login (token with Llama-3.2 access)
from getpass import getpass
from huggingface_hub import login

login(getpass('Enter your Hugging Face token: '))

In [ ]:
# Helper: check which pipeline outputs already exist
from pathlib import Path

checks = {
    'Phase 1 trajectories': Path('results/trajectories.json'),
    'Phase 2 fast scores': Path('results/fast_scores.json'),
    'Phase 3 slow scores': Path('results/slow_scores.json'),
    'Phase 3 correlation': Path('results/correlation.json'),
    'Phase 4 plot': Path('results/importance_by_step.png'),
    'Phase 4 summary': Path('results/poc_summary.md'),
}
for name, path in checks.items():
    print(f"{'OK' if path.exists() else 'MISSING':7}  {name}  ({path})")

In [ ]:
# 5. Phase 1 ONLY — collect trajectories (~5–15 min). Wait for "Saved 10 trajectories".
!python -m agent.runner

In [ ]:
# 6. Phase 2 — attribution patching (fast scores, ~2–5 min)
from pathlib import Path
assert Path('results/trajectories.json').exists(), 'Run cell 5 first and wait for it to finish.'
!python -m interp.attribution_patch

In [ ]:
# 7. Phase 3 — ground-truth patching + correlation (~5–10 min)
from pathlib import Path
assert Path('results/fast_scores.json').exists(), 'Run cell 6 first.'
!python -m interp.ground_truth_patch
!python -m analysis.correlate

In [ ]:
# 8. Phase 4 — plot + summary
from pathlib import Path
assert Path('results/correlation.json').exists(), 'Run cell 7 first.'
!python -m analysis.visualize
!python -m analysis.summarize

In [ ]:
# 9. Show results
import json
from pathlib import Path
from IPython.display import Image, Markdown, display

for p in ['results/correlation.json', 'results/importance_by_step.png', 'results/poc_summary.md']:
    if not Path(p).exists():
        raise FileNotFoundError(f'Missing {p} — run the earlier cells and let each finish.')

print(json.dumps(json.load(open('results/correlation.json')), indent=2))
display(Image('results/importance_by_step.png'))
display(Markdown(open('results/poc_summary.md').read()))